# Rayhan's Task 2 clustering — extracted to run here

Lifted verbatim from `rayhancreated.ipynb` (his `notebooks/main.ipynb` on `main`), so his
result can be reproduced and compared against ours without running anything expensive.

**Only two blocks are needed:** his cleaning (cells 0–19 of the original) and his
clustering (cells 60–81). Everything between them is Task 1 — a TF-IDF baseline, a
HuggingFace model and a DistilBERT fine-tune — and none of it feeds the clustering.

**Cost: under a minute, CPU only.** KMeans over ~38 products. No GPU, no model downloads,
no transformer. The expensive cells were left behind deliberately.

| in the original | here | why |
|---|---|---|
| cells 0–19 | part 1 | builds `df_work`, the only thing clustering needs |
| cells 20–59 | **skipped** | Task 1; cell 52 is `trainer.train()` — hours on this laptop |
| cells 60–81 | part 2 | the clustering itself |
| cells 82–97 | skipped | Task 3 summaries, flan-t5-small |

Nothing in his code was altered except the CSV path, which now resolves in either layout.

## Part 1 — his cleaning (original cells 0–19)

In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

# path resolves in the repo (data/) or in Subha's tree (../datasets/)
CSV = next(c for c in [Path("data/1429_1.csv"),
                       Path("../datasets/1429_1.csv"),
                       Path("datasets/1429_1.csv")] if c.exists())

df = pd.read_csv(CSV, low_memory=False)
print("loaded", CSV, "->", df.shape)

loaded ..\datasets\1429_1.csv -> (34660, 21)


In [3]:
df.head(5)

,id,name,asins,brand,categories,keys,manufacturer,reviews.date,reviews.dateAdded,reviews.dateSeen,...,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,This product so far has not disappointed. My c...,Kindle,NaN,NaN,Adapter
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,great for beginner or experienced person. Boug...,very fast,NaN,NaN,truman
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,Inexpensive tablet for him to use and learn on...,Beginner tablet for our 9 year old son.,NaN,NaN,DaveZ
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,4.0,http://reviews.bestbuy.com/3545/5620406/review...,I've had my Fire HD 8 two weeks now and I love...,Good!!!,NaN,NaN,Shacks
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-12T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,I bought this for my grand daughter when she c...,Fantastic Tablet for kids,NaN,NaN,explore42


In [4]:
df.describe(include='all')

,id,name,asins,brand,categories,keys,manufacturer,reviews.date,reviews.dateAdded,reviews.dateSeen,...,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username
count,34660,27900,34658,34660,34660,34660,34660,34621,24039,34660,...,34066,1.0,34131.000000,34627.000000,34660,34659,34654,0.0,0.0,34653
unique,42,48,41,6,41,42,2,1078,1941,3911,...,2,NaN,NaN,NaN,11929,34659,19766,NaN,NaN,26788
top,AVphgVaX1cnluZ0-DR74,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",B018Y229OU,Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T...",firetablet7displaywifi8gbincludesspecialoffers...,Amazon,2017-01-16T00:00:00.000Z,2017-09-05T22:09:30Z,"2017-09-28T00:00:00Z,2017-09-08T00:00:00Z,2017...",...,True,NaN,NaN,NaN,https://www.amazon.com/product-reviews/B01J2G4...,This product so far has not disappointed. My c...,Great product,NaN,NaN,Mike
freq,10966,10966,10966,28701,10966,10966,34639,710,3127,5072,...,32682,NaN,NaN,NaN,387,1,645,NaN,NaN,138
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,111372787.0,0.630248,4.584573,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,13.215775,0.735653,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,111372787.0,0.000000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,111372787.0,0.000000,4.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,111372787.0,0.000000,5.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,111372787.0,0.000000,5.000000,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34660 entries, 0 to 34659
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    34660 non-null  str    
 1   name                  27900 non-null  str    
 2   asins                 34658 non-null  str    
 3   brand                 34660 non-null  str    
 4   categories            34660 non-null  str    
 5   keys                  34660 non-null  str    
 6   manufacturer          34660 non-null  str    
 7   reviews.date          34621 non-null  str    
 8   reviews.dateAdded     24039 non-null  str    
 9   reviews.dateSeen      34660 non-null  str    
 10  reviews.didPurchase   1 non-null      object 
 11  reviews.doRecommend   34066 non-null  object 
 12  reviews.id            1 non-null      float64
 13  reviews.numHelpful    34131 non-null  float64
 14  reviews.rating        34627 non-null  float64
 15  reviews.sourceURLs    34660 no

In [6]:
print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(15))


Missing values:
reviews.userCity        34660
reviews.userProvince    34660
reviews.id              34659
reviews.didPurchase     34659
reviews.dateAdded       10621
name                     6760
reviews.doRecommend       594
reviews.numHelpful        529
reviews.date               39
reviews.rating             33
reviews.username            7
reviews.title               6
asins                       2
reviews.text                1
categories                  0
dtype: int64


In [7]:
print("\nDuplicate rows:")
print(df.duplicated().sum())


Duplicate rows:
0


In [8]:

print("\nColumns:")
print(df.columns.tolist())


Columns:
['id', 'name', 'asins', 'brand', 'categories', 'keys', 'manufacturer', 'reviews.date', 'reviews.dateAdded', 'reviews.dateSeen', 'reviews.didPurchase', 'reviews.doRecommend', 'reviews.id', 'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs', 'reviews.text', 'reviews.title', 'reviews.userCity', 'reviews.userProvince', 'reviews.username']


In [9]:
df_work = df[
    [
        "name",
        "asins",
        "brand",
        "categories",
        "reviews.rating",
        "reviews.text",
        "reviews.title",
        "reviews.doRecommend"
    ]
].copy()

df_work.columns = [
    "product_name",
    "product_id",
    "brand",
    "category",
    "rating",
    "review_text",
    "review_title",
    "recommended"
]

display(df_work.head())

print("Shape:", df_work.shape)

print("\nMissing values:")
print(df_work.isnull().sum())

print("\nRating distribution:")
print(df_work["rating"].value_counts().sort_index())

print("\nUnique products:")
print(df_work["product_id"].nunique())

,product_name,product_id,brand,category,rating,review_text,review_title,recommended
0,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,This product so far has not disappointed. My c...,Kindle,True
1,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,great for beginner or experienced person. Boug...,very fast,True
2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,Inexpensive tablet for him to use and learn on...,Beginner tablet for our 9 year old son.,True
3,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",4.0,I've had my Fire HD 8 two weeks now and I love...,Good!!!,True
4,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",5.0,I bought this for my grand daughter when she c...,Fantastic Tablet for kids,True


Shape: (34660, 8)

Missing values:
product_name    6760
product_id         2
brand              0
category           0
rating            33
review_text        1
review_title       6
recommended      594
dtype: int64

Rating distribution:
rating
1.0      410
2.0      402
3.0     1499
4.0     8541
5.0    23775
Name: count, dtype: int64

Unique products:
41


In [10]:
# Fill missing product names using another row with the same product_id
name_map = (
    df_work.dropna(subset=["product_name"])
    .drop_duplicates("product_id")
    .set_index("product_id")["product_name"]
)

In [11]:
df_work["product_name"] = df_work["product_name"].fillna(
    df_work["product_id"].map(name_map)
)

In [12]:
# Remove rows where essential information is missing
df_work = df_work.dropna(
    subset=["product_id", "rating", "review_text"]
)

In [13]:
print("Shape after cleaning:", df_work.shape)

print("\nMissing values:")
print(df_work.isnull().sum())

print("\nRating distribution:")
print(df_work["rating"].value_counts().sort_index())

print("\nUnique products:")
print(df_work["product_id"].nunique())

Shape after cleaning: (34624, 8)

Missing values:
product_name    6088
product_id         0
brand              0
category           0
rating             0
review_text        0
review_title       6
recommended      558
dtype: int64

Rating distribution:
rating
1.0      410
2.0      402
3.0     1499
4.0     8540
5.0    23773
Name: count, dtype: int64

Unique products:
38


In [14]:
# Remove duplicate reviews
df_work = df_work.drop_duplicates(
    subset=["product_id", "review_text"]
)

In [15]:
def create_sentiment(rating):
    if rating <= 2:
        return "Negative"
    elif rating == 3:
        return "Neutral"
    else:
        return "Positive"

In [16]:

print("\nRating distribution:")
print(df_work["rating"].value_counts().sort_index())

print("\nUnique products:")
print(df_work["product_id"].nunique())


Rating distribution:
rating
1.0      410
2.0      402
3.0     1499
4.0     8540
5.0    23773
Name: count, dtype: int64

Unique products:
38


In [17]:
df_work["sentiment"] = df_work["rating"].apply(create_sentiment)

In [18]:
print(df_work["sentiment"].value_counts())

sentiment
Positive    32313
Neutral      1499
Negative      812
Name: count, dtype: int64


In [19]:
print(df_work["sentiment"].value_counts(normalize=True) * 100)

sentiment
Positive    93.325439
Neutral      4.329367
Negative     2.345194
Name: proportion, dtype: float64


In [20]:
print("Shape after cleaning:", df_work.shape)

print("\nMissing values:")
print(df_work.isnull().sum())

print("\nRating distribution:")
print(df_work["rating"].value_counts().sort_index())

print("\nUnique products:")
print(df_work["product_id"].nunique())

Shape after cleaning: (34624, 9)

Missing values:
product_name    6088
product_id         0
brand              0
category           0
rating             0
review_text        0
review_title       6
recommended      558
sentiment          0
dtype: int64

Rating distribution:
rating
1.0      410
2.0      402
3.0     1499
4.0     8540
5.0    23773
Name: count, dtype: int64

Unique products:
38


---
## Part 2 — his clustering (original cells 60–81)

Three feature variants in sequence:

- **v1** — product name + category, the straightforward version
- **v2** — same, with `max_features=3000`, evaluated at k = 4, 5 and 6
- **v3** — **the product name repeated four times** plus the category, so the short name
  is not swamped by the long category string

That repetition trick is his answer to the same problem we solved by dropping `categories`
altogether. Two different fixes for one cause; worth comparing rather than assuming either
is right.

Task 02 Clustering

In [21]:
print("Unique products:", df_work["product_id"].nunique())
print("Unique raw categories:", df_work["category"].nunique())

print("\nTop categories:")
print(df_work["category"].value_counts().head(20))

print("\nProducts:")
print(
    df_work[["product_id", "product_name", "brand", "category"]]
    .drop_duplicates("product_id")
    .head(50)
)

Unique products: 38
Unique raw categories: 37

Top categories:
category
Fire Tablets,Tablets,Computers & Tablets,All Tablets,Electronics, Tech Toys, Movies, Music,Electronics,iPad & Tablets,Android Tablets,Frys                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               10966
Stereos,Remote Controls,Amazon Echo,Audio Docks & Mini Speakers,Amazon Echo Accessories,Kitchen & Dining Features,Speaker Systems,Electronics,TVs Entertainment,Clearance,Smart Hubs & Wireless R

In [22]:
product_df = (
    df_work.groupby("product_id")
    .agg(
        product_name=(
            "product_name",
            lambda x: x.dropna().iloc[0]
            if not x.dropna().empty
            else "Unknown"
        ),
        brand=("brand", "first"),
        category=(
            "category",
            lambda x: " ".join(x.dropna().astype(str).unique())
        ),
        review_text=(
            "review_text",
            lambda x: " ".join(x.astype(str).head(50))
        )
    )
    .reset_index()
)

product_df["cluster_text"] = (
    product_df["product_name"].fillna("") + " " +
    product_df["brand"].fillna("") + " " +
    product_df["category"].fillna("") + " " +
    product_df["review_text"].fillna("")
)

print(product_df.shape)

display(
    product_df[
        ["product_id", "product_name", "brand"]
    ].head(40)
)

(38, 6)


,product_id,product_name,brand
0,B002Y27P6Y,Unknown,Amazon Digital Services Inc.
1,B005PB2T0S,"Amazon Kindle Lighted Leather Cover,,,\r\nAmaz...",Amazon
2,B005PB2T2Q,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",Amazon
3,B006GWO5NE,Unknown,Amazon
4,B006GWO5WK,Unknown,Amazon
5,B00DU15MU4,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon
6,B00IOY8XWQ,"Kindle Voyage E-reader, 6 High-Resolution Disp...",Amazon
7,B00IOYAM4I,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon
8,"B00L9EPT8O,B01E6AO69U","Echo (White),,,\r\nEcho (White),,,",Amazon
9,B00LO29KXQ,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon


In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=3000,
    ngram_range=(1, 2)
)

X_products = tfidf.fit_transform(product_df["cluster_text"])

print("TF-IDF shape:", X_products.shape)

TF-IDF shape: (38, 3000)


In [24]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

for k in [4, 5, 6]:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = kmeans.fit_predict(X_products)

    score = silhouette_score(
        X_products,
        labels
    )

    print(f"k={k}, silhouette score={score:.4f}")

k=4, silhouette score=0.0999
k=5, silhouette score=0.1082


k=6, silhouette score=0.1283


In [25]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=6,
    random_state=42,
    n_init=20
)

product_df["cluster"] = kmeans.fit_predict(X_products)

print(product_df["cluster"].value_counts().sort_index())

display(
    product_df[
        ["product_id", "product_name", "brand", "cluster"]
    ].sort_values("cluster")
)

cluster
0    13
1     8
2     4
3     4
4     6
5     3
Name: count, dtype: int64


,product_id,product_name,brand,cluster
24,B018T075DC,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon,0
25,B018Y225IA,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon,0
26,B018Y229OU,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon,0
27,B018Y22BI4,"Echo (White),,,\r\nEcho (White),,,",Amazon,0
28,B018Y22C2Y,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon,0
31,B01AHB9C1E,"Fire HD 8 Tablet with Alexa, 8 HD Display, 32 ...",Amazon,0
29,B018Y23MNM,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",Amazon,0
30,B018Y23P7K,Unknown,Amazon,0
23,B018SZT3BK,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon,0
22,B0189XYY0Q,Unknown,Amazon Fire,0


In [26]:
terms = tfidf.get_feature_names_out()

for cluster_num in range(6):
    center = kmeans.cluster_centers_[cluster_num]
    top_indices = center.argsort()[-15:][::-1]
    top_words = terms[top_indices]

    print(f"\nCluster {cluster_num}")
    print("Top words:", ", ".join(top_words))

    display(
        product_df[
            product_df["cluster"] == cluster_num
        ][
            ["product_id", "product_name", "brand"]
        ]
    )


Cluster 0
Top words: tablet, tablets, great, kids, kindle, games, good, love, use, books, tablets tablets, easy, apps, bought, amazon


,product_id,product_name,brand
16,B00TSUGXKE,"Echo (White),,,\r\nEcho (White),,,",Amazon
22,B0189XYY0Q,Unknown,Amazon Fire
23,B018SZT3BK,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon
24,B018T075DC,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon
25,B018Y225IA,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon
26,B018Y229OU,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon
27,B018Y22BI4,"Echo (White),,,\r\nEcho (White),,,",Amazon
28,B018Y22C2Y,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon
29,B018Y23MNM,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",Amazon
30,B018Y23P7K,Unknown,Amazon



Cluster 1
Top words: kindle, light, paperwhite, reading, voyage, books, read, screen, reader, great, page, easy, love, readers, like


,product_id,product_name,brand
6,B00IOY8XWQ,"Kindle Voyage E-reader, 6 High-Resolution Disp...",Amazon
7,B00IOYAM4I,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon
11,B00OQVZDJM,Amazon Kindle Paperwhite - eBook reader - 4 GB...,Amazon
13,B00QJDU3KY,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon
15,B00REQKWGA,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon
19,B00VINDBJK,Kindle Oasis E-reader with Leather Charging Co...,Amazon
21,B00ZV9PXP2,"Amazon Kindle Fire Hd (3rd Generation) 8gb,,,\...",Amazon
34,B01BFIBRIE,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon



Cluster 2
Top words: cover, kindle, case, leather, stand, covers, closed, light, dx, hinge, like, band, elastic, amazon, kindle cover


...

,product_id,product_name,brand
5,B00DU15MU4,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon
9,B00LO29KXQ,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon
17,B00U3FPN4U,Unknown,Amazon Fire Tv
18,B00UH4D8G2,"Echo (White),,,\r\nEcho (White),,,",Amazon



Cluster 4
Top words: charger, kindle, cord, power, usb, cable, amazon, charging, chargers, plug, works, adapter, cables, charges, charge


,product_id,product_name,brand
3,B006GWO5NE,Unknown,Amazon
4,B006GWO5WK,Unknown,Amazon
12,B00QFQRELG,Unknown,Amazon
14,B00QL1ZN3G,Unknown,Amazon
36,B01J2G4VBG,Amazon 5W USB Official OEM Charger and Power A...,Amazon
37,B01J4ORNHU,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon



Cluster 5
Top words: echo, home, tap, alexa, music, smart, voice, amazon echo, speaker, amazon, fun, smart home, speakers, really, great


,product_id,product_name,brand
8,"B00L9EPT8O,B01E6AO69U","Echo (White),,,\r\nEcho (White),,,",Amazon
20,B00X4WHP5E,Unknown,Amazon
35,B01BH83OOM,Unknown,Amazon Echo


In [27]:
product_df["cluster_text_v2"] = (
    product_df["product_name"].fillna("") + " " +
    product_df["category"].fillna("")
)

tfidf_v2 = TfidfVectorizer(
    stop_words="english",
    max_features=3000,
    ngram_range=(1, 2)
)

X_products_v2 = tfidf_v2.fit_transform(
    product_df["cluster_text_v2"]
)

for k in [4, 5, 6]:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(X_products_v2)

    score = silhouette_score(
        X_products_v2,
        labels
    )

    print(f"k={k}, silhouette score={score:.4f}")

k=4, silhouette score=0.1276


k=5, silhouette score=0.1073


k=6, silhouette score=0.1293


In [28]:
kmeans_v2 = KMeans(
    n_clusters=6,
    random_state=42,
    n_init=20
)

product_df["cluster"] = kmeans_v2.fit_predict(X_products_v2)

print(product_df["cluster"].value_counts().sort_index())

cluster
0    9
1    5
2    4
3    4
4    9
5    7
Name: count, dtype: int64


In [29]:
for cluster in range(6):
    print(f"\n===== CLUSTER {cluster} =====")

    display(
        product_df.loc[
            product_df["cluster"] == cluster,
            ["product_id", "product_name", "brand", "category"]
        ]
    )


===== CLUSTER 0 =====


,product_id,product_name,brand,category
0,B002Y27P6Y,Unknown,Amazon Digital Services Inc.,"Electronics,Amazon Device Accessories,Kindle S..."
1,B005PB2T0S,"Amazon Kindle Lighted Leather Cover,,,\r\nAmaz...",Amazon,"Electronics,eBook Readers & Accessories,Covers..."
2,B005PB2T2Q,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",Amazon,"Amazon Device Accessories,Kindle Store,Kindle ..."
3,B006GWO5NE,Unknown,Amazon,"Power Adapters & Cables,Electronics,USB Cables"
4,B006GWO5WK,Unknown,Amazon,"Computers/Tablets & Networking,Tablet & eBook ..."
10,B00LW9XOJM,Unknown,Amazon,"Cases,Kindle Store,Amazon Device Accessories,A..."
12,B00QFQRELG,Unknown,Amazon,"Chargers & Adapters,Computers & Accessories,Ta..."
14,B00QL1ZN3G,Unknown,Amazon,"Electronics,eBook Readers & Accessories,Power ..."
36,B01J2G4VBG,Amazon 5W USB Official OEM Charger and Power A...,Amazon,"Amazon Devices & Accessories,Amazon Device Acc..."



===== CLUSTER 1 =====


,product_id,product_name,brand,category
5,B00DU15MU4,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon,"Electronics,Categories,Streaming Media Players..."
7,B00IOYAM4I,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon,"Computers & Tablets,E-Readers & Accessories,eB..."
15,B00REQKWGA,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon,"Kindle E-readers,Electronics Features,Computer..."
24,B018T075DC,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T..."
25,B018Y225IA,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon,"Computers/Tablets & Networking,Tablets & eBook..."



===== CLUSTER 2 =====


...

,product_id,product_name,brand,category
9,B00LO29KXQ,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon,"Electronics,Categories,Fire TV,Kindle Store"
13,B00QJDU3KY,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon,"mazon.co.uk,Amazon Devices"
19,B00VINDBJK,Kindle Oasis E-reader with Leather Charging Co...,Amazon,"eBook Readers,Kindle E-readers,Computers & Tab..."
37,B01J4ORNHU,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon,"Kindle Store,Categories,eBook Readers & Access..."



===== CLUSTER 4 =====


,product_id,product_name,brand,category
6,B00IOY8XWQ,"Kindle Voyage E-reader, 6 High-Resolution Disp...",Amazon,"Walmart for Business,Office Electronics,Tablet..."
23,B018SZT3BK,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T..."
26,B018Y229OU,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T..."
28,B018Y22C2Y,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon,"Electronics,Computers,Computer Accessories,Cas..."
29,B018Y23MNM,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",Amazon,"Tablets,Fire Tablets,Computers & Tablets,All T..."
31,B01AHB9C1E,"Fire HD 8 Tablet with Alexa, 8 HD Display, 32 ...",Amazon,"Tablets,Fire Tablets,Computers & Tablets,All T..."
32,B01AHB9CN2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta..."
33,B01AHB9CYG,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Tablets,Fire Tablets,Electronics,Computers,Com..."
34,B01BFIBRIE,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon,"Electronics,iPad & Tablets,All Tablets,Compute..."



===== CLUSTER 5 =====


,product_id,product_name,brand,category
11,B00OQVZDJM,Amazon Kindle Paperwhite - eBook reader - 4 GB...,Amazon,"Walmart for Business,Office Electronics,Tablet..."
16,B00TSUGXKE,"Echo (White),,,\r\nEcho (White),,,",Amazon,"Electronics Features,Fire Tablets,Computers & ..."
18,B00UH4D8G2,"Echo (White),,,\r\nEcho (White),,,",Amazon,"Categories,Streaming Media Players,Electronics"
21,B00ZV9PXP2,"Amazon Kindle Fire Hd (3rd Generation) 8gb,,,\...",Amazon,"Electronics,iPad & Tablets,All Tablets,Compute..."
22,B0189XYY0Q,Unknown,Amazon Fire,"Electronics,Tablets & E-Readers,Tablets,Back T..."
27,B018Y22BI4,"Echo (White),,,\r\nEcho (White),,,",Amazon,"Computers & Tablets,Tablets,All Tablets,Comput..."
30,B018Y23P7K,Unknown,Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T..."


In [30]:
terms = tfidf_v2.get_feature_names_out()

centers = kmeans_v2.cluster_centers_.argsort()[:, ::-1]

for cluster in range(6):
    top_words = [
        terms[i]
        for i in centers[cluster, :15]
    ]

    print(f"Cluster {cluster}:")
    print(", ".join(top_words))
    print()

Cluster 0:
accessories, kindle, cables, accessories kindle, power, generation, adapters, amazon, device, device accessories, amazon device, unknown, power adapters, kindle store, store

Cluster 1:
tablets, blue, display tablet, brand new, brand, ips display, tablet wifi, 16gb, new amazon, gb blue, ips, kindle 16gb, 16gb ips, wifi 16, wifi

Cluster 2:
home, smart, speakers, smart home, audio, hubs, electronics, tvs, devices, home theater, theater, smart hubs, security, automation, home automation

Cluster 3:
offers black, includes, includes special, fi gb, black, offers, special offers, special, wi fi, wi, fi, gb includes, tablet display, display wi, readers

Cluster 4:
tablets, tablets tablets, hd, tablets computers, computers, wi fi, wi, fi, special, special offers, offers, display wi, computers tablets, display, includes

Cluster 5:
tablets, echo white, white, echo, tablets tablets, computers, readers, computers tablets, tablets computers, electronics, white echo, tablets frys, ebook

In [31]:
kmeans_4 = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=20
)

product_df["cluster_4"] = kmeans_4.fit_predict(X_products_v2)

print(
    product_df["cluster_4"]
    .value_counts()
    .sort_index()
)

cluster_4
0    18
1    10
2     5
3     5
Name: count, dtype: int64


In [32]:
for cluster in range(4):
    print(f"\n===== CLUSTER {cluster} =====")

    cluster_products = product_df.loc[
        product_df["cluster_4"] == cluster,
        ["product_id", "product_name", "brand"]
    ]

    display(cluster_products)


===== CLUSTER 0 =====


,product_id,product_name,brand
6,B00IOY8XWQ,"Kindle Voyage E-reader, 6 High-Resolution Disp...",Amazon
9,B00LO29KXQ,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon
11,B00OQVZDJM,Amazon Kindle Paperwhite - eBook reader - 4 GB...,Amazon
13,B00QJDU3KY,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon
16,B00TSUGXKE,"Echo (White),,,\r\nEcho (White),,,",Amazon
19,B00VINDBJK,Kindle Oasis E-reader with Leather Charging Co...,Amazon
21,B00ZV9PXP2,"Amazon Kindle Fire Hd (3rd Generation) 8gb,,,\...",Amazon
22,B0189XYY0Q,Unknown,Amazon Fire
23,B018SZT3BK,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon
26,B018Y229OU,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon



===== CLUSTER 1 =====


,product_id,product_name,brand
0,B002Y27P6Y,Unknown,Amazon Digital Services Inc.
1,B005PB2T0S,"Amazon Kindle Lighted Leather Cover,,,\r\nAmaz...",Amazon
2,B005PB2T2Q,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",Amazon
3,B006GWO5NE,Unknown,Amazon
4,B006GWO5WK,Unknown,Amazon
10,B00LW9XOJM,Unknown,Amazon
12,B00QFQRELG,Unknown,Amazon
14,B00QL1ZN3G,Unknown,Amazon
36,B01J2G4VBG,Amazon 5W USB Official OEM Charger and Power A...,Amazon
37,B01J4ORNHU,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon



===== CLUSTER 2 =====


,product_id,product_name,brand
5,B00DU15MU4,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon
7,B00IOYAM4I,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon
15,B00REQKWGA,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon
24,B018T075DC,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon
25,B018Y225IA,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon



===== CLUSTER 3 =====


,product_id,product_name,brand
8,"B00L9EPT8O,B01E6AO69U","Echo (White),,,\r\nEcho (White),,,",Amazon
17,B00U3FPN4U,Unknown,Amazon Fire Tv
18,B00UH4D8G2,"Echo (White),,,\r\nEcho (White),,,",Amazon
20,B00X4WHP5E,Unknown,Amazon
35,B01BH83OOM,Unknown,Amazon Echo


In [33]:
terms = tfidf_v2.get_feature_names_out()

centers_4 = kmeans_4.cluster_centers_.argsort()[:, ::-1]

for cluster in range(4):
    top_words = [
        terms[i]
        for i in centers_4[cluster, :15]
    ]

    print(f"Cluster {cluster}:")
    print(", ".join(top_words))
    print()

Cluster 0:
tablets, tablets tablets, computers, tablets computers, computers tablets, readers, electronics, wi, fi, wi fi, special offers, offers, special, includes, includes special

Cluster 1:
accessories, kindle, cables, power, adapters, accessories kindle, power adapters, device accessories, amazon device, device, amazon, generation, kindle store, store, adapters cables

Cluster 2:
tablets, blue, display tablet, brand new, brand, ips display, tablet wifi, 16gb, new amazon, gb blue, ips, kindle 16gb, 16gb ips, wifi 16, wifi

Cluster 3:
home, smart, echo, speakers, echo white, white, smart home, audio, electronics, streaming, media, players, streaming media, media players, hubs



In [34]:
name_text = (
    product_df["product_name"]
    .replace("Unknown", "")
    .fillna("")
)

product_df["cluster_text_v3"] = (
    (name_text + " ") * 4
    + product_df["category"].fillna("")
)

In [35]:
tfidf_v3 = TfidfVectorizer(
    stop_words="english",
    max_features=3000,
    ngram_range=(1, 2)
)

X_products_v3 = tfidf_v3.fit_transform(
    product_df["cluster_text_v3"]
)

In [36]:
for k in [4, 5, 6]:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(X_products_v3)

    score = silhouette_score(
        X_products_v3,
        labels
    )

    print(f"k={k}, silhouette score={score:.4f}")

k=4, silhouette score=0.2313


k=5, silhouette score=0.2601


k=6, silhouette score=0.2673


In [37]:
kmeans_v3 = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=20
)

product_df["cluster_v3"] = kmeans_v3.fit_predict(
    X_products_v3
)

for cluster in range(4):
    print(f"\n===== CLUSTER {cluster} =====")

    display(
        product_df.loc[
            product_df["cluster_v3"] == cluster,
            ["product_id", "product_name"]
        ]
    )


===== CLUSTER 0 =====


,product_id,product_name
0,B002Y27P6Y,Unknown
1,B005PB2T0S,"Amazon Kindle Lighted Leather Cover,,,\r\nAmaz..."
3,B006GWO5NE,Unknown
4,B006GWO5WK,Unknown
10,B00LW9XOJM,Unknown
12,B00QFQRELG,Unknown
14,B00QL1ZN3G,Unknown
17,B00U3FPN4U,Unknown
21,B00ZV9PXP2,"Amazon Kindle Fire Hd (3rd Generation) 8gb,,,\..."
22,B0189XYY0Q,Unknown



===== CLUSTER 1 =====


,product_id,product_name
2,B005PB2T2Q,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16..."
6,B00IOY8XWQ,"Kindle Voyage E-reader, 6 High-Resolution Disp..."
9,B00LO29KXQ,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes..."
11,B00OQVZDJM,Amazon Kindle Paperwhite - eBook reader - 4 GB...
13,B00QJDU3KY,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes..."
19,B00VINDBJK,Kindle Oasis E-reader with Leather Charging Co...
23,B018SZT3BK,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes..."
26,B018Y229OU,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes..."
28,B018Y22C2Y,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes..."
29,B018Y23MNM,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16..."



===== CLUSTER 2 =====


,product_id,product_name
5,B00DU15MU4,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...
7,B00IOYAM4I,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...
15,B00REQKWGA,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...
24,B018T075DC,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...
25,B018Y225IA,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...



===== CLUSTER 3 =====


,product_id,product_name
8,"B00L9EPT8O,B01E6AO69U","Echo (White),,,\r\nEcho (White),,,"
16,B00TSUGXKE,"Echo (White),,,\r\nEcho (White),,,"
18,B00UH4D8G2,"Echo (White),,,\r\nEcho (White),,,"
20,B00X4WHP5E,Unknown
27,B018Y22BI4,"Echo (White),,,\r\nEcho (White),,,"
30,B018Y23P7K,Unknown
35,B01BH83OOM,Unknown


In [38]:
kmeans_6_v3 = KMeans(
    n_clusters=6,
    random_state=42,
    n_init=20
)

product_df["cluster_v3_6"] = kmeans_6_v3.fit_predict(
    X_products_v3
)

print(
    product_df["cluster_v3_6"]
    .value_counts()
    .sort_index()
)

cluster_v3_6
0    11
1     7
2     5
3     7
4     6
5     2
Name: count, dtype: int64


In [39]:
for cluster in range(6):
    print(f"\n===== CLUSTER {cluster} =====")

    display(
        product_df.loc[
            product_df["cluster_v3_6"] == cluster,
            ["product_id", "product_name"]
        ]
    )


===== CLUSTER 0 =====


,product_id,product_name
0,B002Y27P6Y,Unknown
1,B005PB2T0S,"Amazon Kindle Lighted Leather Cover,,,\r\nAmaz..."
3,B006GWO5NE,Unknown
4,B006GWO5WK,Unknown
10,B00LW9XOJM,Unknown
11,B00OQVZDJM,Amazon Kindle Paperwhite - eBook reader - 4 GB...
12,B00QFQRELG,Unknown
14,B00QL1ZN3G,Unknown
17,B00U3FPN4U,Unknown
22,B0189XYY0Q,Unknown



===== CLUSTER 1 =====


,product_id,product_name
9,B00LO29KXQ,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes..."
13,B00QJDU3KY,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes..."
23,B018SZT3BK,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes..."
26,B018Y229OU,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes..."
28,B018Y22C2Y,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes..."
34,B01BFIBRIE,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes..."
37,B01J4ORNHU,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes..."



===== CLUSTER 2 =====


...

,product_id,product_name
8,"B00L9EPT8O,B01E6AO69U","Echo (White),,,\r\nEcho (White),,,"
16,B00TSUGXKE,"Echo (White),,,\r\nEcho (White),,,"
18,B00UH4D8G2,"Echo (White),,,\r\nEcho (White),,,"
20,B00X4WHP5E,Unknown
27,B018Y22BI4,"Echo (White),,,\r\nEcho (White),,,"
30,B018Y23P7K,Unknown
35,B01BH83OOM,Unknown



===== CLUSTER 4 =====


,product_id,product_name
2,B005PB2T2Q,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16..."
21,B00ZV9PXP2,"Amazon Kindle Fire Hd (3rd Generation) 8gb,,,\..."
29,B018Y23MNM,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16..."
31,B01AHB9C1E,"Fire HD 8 Tablet with Alexa, 8 HD Display, 32 ..."
32,B01AHB9CN2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,..."
33,B01AHB9CYG,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,..."



===== CLUSTER 5 =====


,product_id,product_name
6,B00IOY8XWQ,"Kindle Voyage E-reader, 6 High-Resolution Disp..."
19,B00VINDBJK,Kindle Oasis E-reader with Leather Charging Co...


In [40]:
terms_v3 = tfidf_v3.get_feature_names_out()

centers_v3 = kmeans_6_v3.cluster_centers_.argsort()[:, ::-1]

for cluster in range(6):
    top_words = [
        terms_v3[i]
        for i in centers_v3[cluster, :15]
    ]

    print(f"Cluster {cluster}:")
    print(", ".join(top_words))
    print()

Cluster 0:
accessories, kindle, cables, power, amazon, adapters, electronics, college, tablets, tablet accessories, accessories tablet, power adapters, accessories kindle, device accessories, device

Cluster 1:
fi gb, tablet display, gb includes, offers black, black, includes, display wi, includes special, offers, special, special offers, wi, fi, wi fi, black tablet

Cluster 2:
kindle 16gb, brand, tablet wifi, new amazon, ips display, brand new, ips, display tablet, gb blue, 16gb, 16gb ips, wifi 16, wifi, blue, new

Cluster 3:
echo white, white, echo, white echo, home, tablets, smart, tablets tablets, speakers, frys, smart home, tablets frys, audio, hubs, tablets computers

Cluster 4:
hd, fi 16, hd tablet, hd display, display wi, gb, display, tablet, fi, wi, wi fi, tablets, 16, 16 gb, tablet hd

Cluster 5:
300 ppi, 300, high resolution, display 300, fi includes, resolution, resolution display, ppi, high, voyage, kindle voyage, offers kindle, wi fi, wi, fi



In [41]:
cluster_names = {
    0: "Accessories & Power",
    1: "Fire Tablets - Standard",
    2: "Kindle Fire 16GB Tablets",
    3: "Echo & Smart Home",
    4: "Fire HD Tablets",
    5: "Kindle Voyage / E-readers"
}

product_df["meta_category"] = (
    product_df["cluster_v3_6"]
    .map(cluster_names)
)

display(
    product_df[
        [
            "product_id",
            "product_name",
            "cluster_v3_6",
            "meta_category"
        ]
    ].sort_values("meta_category")
)

,product_id,product_name,cluster_v3_6,meta_category
0,B002Y27P6Y,Unknown,0,Accessories & Power
1,B005PB2T0S,"Amazon Kindle Lighted Leather Cover,,,\r\nAmaz...",0,Accessories & Power
3,B006GWO5NE,Unknown,0,Accessories & Power
4,B006GWO5WK,Unknown,0,Accessories & Power
10,B00LW9XOJM,Unknown,0,Accessories & Power
11,B00OQVZDJM,Amazon Kindle Paperwhite - eBook reader - 4 GB...,0,Accessories & Power
12,B00QFQRELG,Unknown,0,Accessories & Power
14,B00QL1ZN3G,Unknown,0,Accessories & Power
17,B00U3FPN4U,Unknown,0,Accessories & Power
22,B0189XYY0Q,Unknown,0,Accessories & Power


---
## What to check in the output

**How many products are named `Unknown`.** His saved run has **11 of 38**, because he
groups by `product_id` and many ids carry no name. Nine of those eleven land in
`Accessories & Power`, which makes that cluster largely a bucket of unidentified products
rather than a category.

**Whether the names are still corrupted.** Watch for `Echo (White),,,\r\nEcho (White),,,`
— he did not apply the name repair, so the duplication goes straight into the clustering
text.

**How many distinct products he actually has.** 38 `product_id` values collapse to **15
distinct names**: six separate ids all called "Fire Tablet, 7 Display… Black", five
"Kindle Fire 16gb… Blue", four "Echo (White)". Our Step 0 established that `id`/`asins`
is not a reliable product key, and this is what that looks like downstream.

### The one-line improvement

In his `product_df = df_work.groupby("product_id")`, group by **`product_name`** instead —
using the repaired names from our Step 0. That alone should empty the `Unknown` bucket and
take the product count from 38 ambiguous ids to 40 real products.